<a href="https://colab.research.google.com/github/MatteoBaraldi/Machine-Learning-for-Bioengineering/blob/main/MOD-1/exams-templates/Template_ML4Bioengineering%2020260622.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
## Practical exercise
# The code below trains and evaluates a decision tree classifier.
# The analysis contains mistakes and choices that could be improved. Answer to the following questions and correct the code accordingly:

# 1) Is the estimate of the accuracy in the original code biased ? Briefly explain why. You can write the answer to this and the following questions as comments in the code.
#    Answer 1: Yes, the estimate of the accuracy in the original code (before correction) was biased.
#    This is because the model's performance was evaluated on the entire dataset (X), which included the training data.
#    Evaluating on data the model has already seen leads to an overly optimistic and unrealistic estimate of its generalization performance on new, unseen data.

# 2) Implement a strategy to reduce the risk of overfitting.
#    Answer 2 & Implementation: Strategies implemented to reduce overfitting include:
#    - Using `stratify=y` in `train_test_split`: Ensures that the proportion of target classes is maintained in both training and test sets, which is good practice, especially for imbalanced datasets.
#    - Setting `random_state`: Ensures reproducibility of the splits and model training.
#    - Parameter tuning with `GridSearchCV`:
#      - `max_depth`: Limiting the maximum depth of the tree (e.g., `max_depth = 3`) prevents the tree from becoming too complex and memorizing the training data.
#      - `ccp_alpha`: Cost-complexity pruning (ccp_alpha) is used to prune the tree by removing the weakest links based on a complexity parameter. A range of `ccp_alpha` values is explored to find the optimal pruning level.
#    - Evaluating on a separate `X_test` set: The final accuracy is reported on `y_test` and `y_pred_test_pruned` to provide an unbiased estimate of the model's performance on unseen data.

# 3) Is the importance of the features accurately estimated ? Briefly explain why.
#    Answer 3: No, the importance of features estimated from a single, potentially overfit Decision Tree (as in the original code) is often not accurately estimated or stable.
#    A single decision tree can be very sensitive to small changes in the training data, leading to high variance in feature importance scores.
#    If the tree overfits, its reported feature importances might reflect spurious correlations in the training data rather than true underlying relationships.

# 4) Implement a possible strategy to improve the estimate of the features' accuracy
#    Answer 4 & Implementation: The implemented strategy to improve feature importance estimation is to use a **Random Forest Classifier**.
#    Random Forests build an ensemble of multiple decision trees. The feature importance from a Random Forest is calculated by averaging the importance (e.g., based on Gini impurity reduction) across all trees in the forest.
#    This averaging process significantly reduces the variance associated with individual trees, providing a more robust, stable, and accurate estimate of feature importance.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Split data into training and testing sets, ensuring stratification and reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Define the parameter grid for GridSearchCV, including max_depth and ccp_alpha for pruning
grid = {
    "max_depth": [3],
    "min_samples_leaf": [10,],
    "ccp_alpha": [0.0, 0.001, 0.005, 0.01, 0.02],
}

# Initialize Decision Tree Classifier with a random state for reproducibility
dt_cv =  DecisionTreeClassifier(random_state= 42)

# Setup GridSearchCV for hyperparameter tuning with 5-fold cross-validation
search = GridSearchCV(
    estimator = dt_cv,
    param_grid=grid,
    scoring="balanced_accuracy", # Use balanced_accuracy for potential class imbalance
    cv=5,
    n_jobs=-1,
    return_train_score=True
)

# Fit GridSearchCV on the training data
search.fit(X_train, y_train)

# Get the best estimator (pruned Decision Tree model)
best_model = search.best_estimator_

# Predict on training and test sets using the best model
y_pred_train_pruned = best_model.predict(X_train)
y_pred_test_pruned = best_model.predict(X_test)

# Print accuracy on both training and test sets to assess generalization
print("Training Accuracy:", accuracy_score(y_train, y_pred_train_pruned))
print("Test Accuracy:", accuracy_score(y_test, y_pred_test_pruned))

# Train a RandomForestClassifier to get more robust feature importances
rf = RandomForestClassifier(
    n_estimators=100, max_depth = 3, random_state= 42)
rf.fit(X_train, y_train)

# Get and display feature importances from the Random Forest model
importances = pd.Series(rf.feature_importances_, index=X.columns)
print("\nTop 10 Feature Importances from Random Forest:")
print(importances.sort_values(ascending=False).head(10))
